## Heart Disease Prediction 

In [2]:
!pip install pandas numpy scikit-learn matplotlib seaborn joblib


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\LAPTOP LAb\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


### Imports and folder check

In [ ]:
%matplotlib inline
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GroupShuffleSplit, GridSearchCV, LeaveOneGroupOut, cross_val_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, RocCurveDisplay,
                             ConfusionMatrixDisplay)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass

DATA_PATH = "../data/heart_extended.csv"
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "heart_disease_model.joblib")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Notebook folder:", os.getcwd())
print("Data file found:", os.path.exists(DATA_PATH))
print("Model folder:   ", os.path.abspath(MODEL_DIR))

 ### Step 2: Load the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", df.shape)
print("Unique cohort rows:", df["source"].value_counts().to_dict())
df.head()

### Step 3: Inspect the data 

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())
df.describe().T

### Step 4: Remove duplicate rows

In [ ]:
print("Duplicate rows before:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)
print(df["target"].value_counts())

### Exploratory data analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
 
sns.countplot(x="target", data=df, ax=axes[0])
axes[0].set_title("Heart disease distribution")
 
sns.histplot(data=df, x="age", hue="target", kde=True, ax=axes[1])
axes[1].set_title("Age by target")
 
sns.histplot(data=df, x="thalach", hue="target", kde=True, ax=axes[2])
axes[2].set_title("Max heart rate by target")
 
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation heatmap")
plt.show()

### Step 6: Prepare features and split the data

In [ ]:
cat_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numeric_cols = [column for column in df.columns if column not in cat_cols + ["target", "source"]]

X = df.drop(columns=["target", "source"])
y = df["target"].astype(int)
groups = df["source"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print("Number of raw features:", X.shape[1])
print("Train size:", X_train.shape, " Test size:", X_test.shape)
print("Held-out test cohort(s):", sorted(groups.iloc[test_idx].unique()))

### Step 7: Compare the 4 models with cross-validation

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
])

def make_model(classifier):
    return Pipeline([
        ("preprocessor", preprocessor),
        ("clf", classifier),
    ])

models = {
    "Logistic Regression": make_model(
        LogisticRegression(max_iter=2000)
    ),
    "Naive Bayes": make_model(GaussianNB()),
    "Decision Tree": make_model(DecisionTreeClassifier(random_state=42)),
    "kNN": make_model(KNeighborsClassifier(n_neighbors=5)),
    "Random Forest": make_model(
        RandomForestClassifier(random_state=42)
    ),
    "Extra Trees": make_model(
        ExtraTreesClassifier(random_state=42)
    ),
}

cv = LeaveOneGroupOut()

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, groups=groups_train, cv=cv, scoring="accuracy"
    )
    print(f"{name:20s} leave-one-cohort-out accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

### Step 8: Hyperparameter tuning

In [ ]:
param_grids = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10],
                            "clf__class_weight": [None, "balanced"]},
    "Naive Bayes": {"clf__var_smoothing": np.logspace(-9, -1, 9)},
    "Decision Tree": {"clf__max_depth": [2, 3, 4, 5, 6, 8, None],
                      "clf__min_samples_leaf": [1, 2, 5, 10]},
    "kNN": {"clf__n_neighbors": list(range(1, 31, 2)),
            "clf__weights": ["uniform", "distance"]},
    "Random Forest": {"clf__n_estimators": [100, 300],
                      "clf__max_depth": [None, 8],
                      "clf__min_samples_leaf": [1, 4],
                      "clf__class_weight": [None, "balanced"]},
    "Extra Trees": {"clf__n_estimators": [100, 300],
                    "clf__max_depth": [None, 8],
                    "clf__min_samples_leaf": [1, 4, 8],
                    "clf__class_weight": [None, "balanced"]},
}

best_models = {}

for name, grid in param_grids.items():
    gs = GridSearchCV(models[name], grid, cv=cv, scoring="accuracy", n_jobs=1)
    gs.fit(X_train, y_train, groups=groups_train)
    best_models[name] = gs.best_estimator_
    print(f"{name:20s} best params: {gs.best_params_}  leave-one-cohort-out accuracy: {gs.best_score_:.3f}")

### Step 9: Final evaluation on the test set

In [ ]:
def evaluate(name, model):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
    }
 
results = pd.DataFrame([evaluate(n, m) for n, m in best_models.items()])
results.sort_values("Accuracy", ascending=False).round(3)


### Step 10: Visual comparison and the best model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in best_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC curves on the test set")
ax.legend()
plt.show()

In [ ]:
best_name = results.sort_values("Accuracy", ascending=False).iloc[0]["Model"]
best_model = best_models[best_name]
print("Best model:", best_name)
 
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test,
                                      display_labels=["No disease", "Disease"],
                                      cmap="Blues")
plt.title(f"Confusion matrix: {best_name}")
plt.show()
 
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=["No disease", "Disease"]))

### Step 11: What did the model learn?

In [ ]:
if best_name == "Logistic Regression":
    coefs = pd.Series(best_model.named_steps["clf"].coef_[0], index=X.columns)
    coefs.sort_values().plot(kind="barh", figsize=(8, 8), title="Logistic Regression coefficients")
    plt.show()
 
elif best_name == "Decision Tree":
    plt.figure(figsize=(20, 10))
    plot_tree(best_model, feature_names=X.columns, class_names=["No", "Disease"],
              filled=True, max_depth=3, fontsize=9)
    plt.show()
 
else:
    print(f"{best_name} has no coefficients or tree to display.")


### Step 12: Save the model

In [ ]:
best_model.fit(X, y)

bundle = {
    "model": best_model,
    "model_name": best_name,
    "features": list(X.columns),
    "cat_cols": cat_cols,
    "input_mode": "raw",
}

joblib.dump(bundle, MODEL_PATH)

if os.path.exists(MODEL_PATH):
    print(f"Saved OK: {os.path.abspath(MODEL_PATH)}")
    print(f"File size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")
else:
    raise FileNotFoundError(f"Model was not created: {MODEL_PATH}")

### Step 13: Load the saved model and predict for a new patient

In [ ]:
def make_features(patient, features, cat_cols):
    row = {f: 0 for f in features}
    for col, value in patient.items():
        if col in cat_cols:
            key = f"{col}_{int(value)}"
            if key in row:
                row[key] = 1
        elif col in row:
            row[col] = value
    return pd.DataFrame([row])[features]
 
 
loaded = joblib.load(MODEL_PATH)
model = loaded["model"]
features = loaded["features"]
cat_cols_loaded = loaded["cat_cols"]
 
print("Loaded model:", loaded["model_name"])
 
# Check: encoding a real training patient must match the training matrix
sample = df.drop(columns="target").iloc[0].to_dict()
check = make_features(sample, features, cat_cols_loaded)
print("Encoding matches training data:",
      np.allclose(check.values[0].astype(float), X.iloc[0].values.astype(float)))
new_patient = {
    "age": 58, "sex": 1, "cp": 0, "trestbps": 140, "chol": 250,
    "fbs": 0, "restecg": 1, "thalach": 130, "exang": 1, "oldpeak": 2.0,
    "slope": 1, "ca": 1, "thal": 2
}
 
X_new = make_features(new_patient, features, cat_cols_loaded)
prob = model.predict_proba(X_new)[0, 1]
pred = model.predict(X_new)[0]
 
print(f"Predicted: {'Heart disease' if pred == 1 else 'No heart disease'}")
print(f"Probability of disease: {prob:.1%}")

In [ ]:
import joblib
import pandas as pd

bundle = joblib.load("../models/heart_disease_model.joblib")
model = bundle["model"]
features = bundle["features"]

fields = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
          "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

print("Enter values (sex: 1=male, 0=female | fbs: 1 or 0 | exang: 1 or 0)")
patient = {}
for field in fields:
    patient[field] = float(input(f"{field}: "))

X_new = pd.DataFrame([patient], columns=features)
prob = model.predict_proba(X_new)[0, 1]
pred = model.predict(X_new)[0]

print()
print(f"Prediction: {'Heart disease' if pred == 1 else 'No heart disease'}")
print(f"Probability of disease: {prob:.1%}")

In [ ]:
import joblib
import pandas as pd

bundle = joblib.load("../models/heart_disease_model.joblib")
model = bundle["model"]
features = bundle["features"]
raw = pd.read_csv(DATA_PATH)

fields = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
          "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

allowed_values = {
    "sex": [0, 1], "cp": [0, 1, 2, 3], "fbs": [0, 1],
    "restecg": [0, 1, 2], "exang": [0, 1], "slope": [0, 1, 2],
    "ca": [0, 1, 2, 3], "thal": [1, 2, 3],
}
guide = {
    "age": "Age in years",
    "sex": "1 = male, 0 = female",
    "cp": "Chest pain type: 0 typical, 1 atypical, 2 non-anginal, 3 asymptomatic",
    "trestbps": "Resting blood pressure (mm Hg)",
    "chol": "Serum cholesterol (mg/dl)",
    "fbs": "Fasting blood sugar > 120 mg/dl: 1 = yes, 0 = no",
    "restecg": "Resting ECG: 0 normal, 1 ST-T abnormality, 2 LV hypertrophy",
    "thalach": "Maximum heart rate achieved",
    "exang": "Exercise-induced chest pain: 1 = yes, 0 = no",
    "oldpeak": "ST depression from exercise (decimals allowed, e.g. 1.5)",
    "slope": "ST slope: 0 upsloping, 1 flat, 2 downsloping",
    "ca": "Number of major vessels colored (0 to 3)",
    "thal": "Thalassemia: 1 normal, 2 fixed defect, 3 reversible defect",
}

print("Input reference:")
for field in fields:
    if field in allowed_values:
        print(f"  {field:9s} {guide[field]} -> allowed: {allowed_values[field]}")
    else:
        print(f"  {field:9s} {guide[field]} -> range in data: {raw[field].min()} to {raw[field].max()}")
print()

def ask(field):
    if field in allowed_values:
        allowed = allowed_values[field]
        prompt = f"{field} ({guide[field]}) allowed {allowed}: "
    else:
        lo, hi = raw[field].min(), raw[field].max()
        prompt = f"{field} ({guide[field]}) range {lo} to {hi}: "

    while True:
        text = input(prompt).strip()
        try:
            value = float(text)
        except ValueError:
            print("  Please type a number.")
            continue

        if field in allowed_values:
            if value != int(value) or int(value) not in allowed:
                print(f"  Not allowed. Choose one of {allowed}.")
                continue
        elif not (lo <= value <= hi):
            print(f"  Enter a value between {lo} and {hi}.")
            continue
        return value

patient = {field: ask(field) for field in fields}
X_new = pd.DataFrame([patient], columns=features)
prob = model.predict_proba(X_new)[0, 1]
pred = model.predict(X_new)[0]

print()
print(f"Prediction: {'Heart disease' if pred == 1 else 'No heart disease'}")
print(f"Probability of disease: {prob:.1%}")